# Weight Notebook

[Colab Link](https://colab.research.google.com/github/casangi/astroviper/blob/main/docs/core_tutorials/imaging/briggs_imaging_weights.ipynb)

The purpose of this notebook is to demonstrate that there is agreement between CASA and AstroVIPER when calculating imaging weights using Briggs weighting with robust values of [-2, -1.45, -0.5, 0.0, 0.5, 1.33, 2].  
Three single-field use cases will be considered: single polarization (XX), parallel hands (XX, YY), and full polarization (XX, XY, YX, YY).  
Per-channel weight density is always `True`.

This notebook demonstrates only the serial core functions and does not include any parallelization.

These functions are all low level functions and not intended as the final external user interface. For example the functions expect all inputs in standard SI units and will not do any units conversion.

---

## How CASA Calculates Imaging Weights

The same weights are applied to all polarizations:

- **Single Polarization (XX):** Imaging weights are calculated using the XX data weights.  
- **Parallel Hands (XX, YY):** Imaging weights are calculated using the average of the XX and YY data weights.  
- **Full Polarization (XX, XY, YX, YY):** Imaging weights are calculated using the average of the XX and YY data weights and applied to all polarizations (XX, XY, YX, YY).  
  *Note: The XY and YX data weights are ignored.*

---

## Pseudo Code

ps_xdt Processing Set Data Tree
ms_xdt Measurement Set v4 Data Tree
```
core/imaging/calculate_imaging_weights(ps_xdt)
    for ms_xdt in ps_xdt
        #Grid the averaged data weights
        grid_imaging_weights(weight_density_grid,ms_xdt)

    #Calculate Briggs Parameters
    briggs_factors = calculate_briggs_params(weight_density_grid)
    
    #Degrid and applies Briggs parameter
    for ms_xdt in ps_xdt:
        imaging_weights = degrid_imaging_weights(weight_density_grid)
```

---

## Notes

- `mosweight` has not been implemented yet. This is planned for future work.  
- `perchanweightdensity` is always `True`.  
- `briggsbwtaper` has not been implemented yet.  
- CASA interpolation was changed from *linear* to *nearest*.  

---

## Questions

- Should we continue handling the weights in the same way as CASA, or should we consider a more sophisticated approach for the full polarization case?  
- Are there any function or parameter names we should change?  
- Are there missing weighting options that we should demonstrate?  
- What interpolation method should be used when data channel frequencies differ from image channel frequencies?  
- Should we calculate the weights independently for each execution block or scheduling block?  
  *Example: the last full polarization case (`3c286_Band6_5chans_lsrk_compare_weights.ps.zarr`) has three execution blocks from the same scheduling block.*

## Install AstroVIPER

Skip this cell if you don't want to install the latest version of AstroVIPER.

In [ ]:
import os
from importlib.metadata import version

try:
    os.system("pip install --upgrade astroviper")

    import astroviper

    print("Using astroviper version", version("astroviper"))

except ImportError as exc:
    print(f"Could not import astroviper: {exc}")

## API

In [ ]:
from astroviper.processing_functions.imaging.calculate_imaging_weights import (
    calculate_imaging_weights,
)

calculate_imaging_weights?

## Single Pol XX

In [ ]:
from toolviper.utils.data import download, update
from xradio.measurement_set import open_processing_set

update()
download(file="twhya_selfcal_5chans_lsrk_xx_compare_weights.ps.zarr")
ps_single_pol_xdt = open_processing_set(
    "twhya_selfcal_5chans_lsrk_xx_compare_weights.ps.zarr"
)
ps_single_pol_xdt.xr_ps.summary()

In [ ]:
(
    ps_single_pol_xdt["twhya_selfcal_5chans_lsrk_xx_0"].frequency[1]
    - ps_single_pol_xdt["twhya_selfcal_5chans_lsrk_xx_0"].frequency[0]
)

### Create Weights With AstroVIPER

In [ ]:
import numpy as np
from xradio.image import make_empty_sky_image

from astroviper.processing_functions.imaging.calculate_imaging_weights import (
    calculate_imaging_weights,
)

image_size = [250, 250]
cell_size = np.array([-0.1, 0.1]) * np.pi / (180 * 3600)

# The empty sky image only defines the uv-grid geometry used to build the
# Briggs weight-density grid.  calculate_imaging_weights writes the imaging
# weights back into the processing set in place, one data group per robust
# value (out group "av<robust>", variable WEIGHT_AV_R_<robust>).
combined = ps_single_pol_xdt.xr_ps.get_combined_field_and_source_xds()
phase_center = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=combined.attrs["center_field_name"]
).values
frequency_coords = ps_single_pol_xdt.xr_ps.get_freq_axis().values
pol_coords = list(
    ps_single_pol_xdt["twhya_selfcal_5chans_lsrk_xx_0"].polarization.values
)

img_xds = make_empty_sky_image(
    phase_center=phase_center,
    image_size=image_size,
    cell_size=cell_size,
    frequency_coords=frequency_coords,
    pol_coords=pol_coords,
    time_coords=[0],
    do_sky_coords=False,
)
img_xds.attrs["type"] = "image_dataset"

for robust in [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]:
    print("Calculating robust =", robust)
    calculate_imaging_weights(
        ps_single_pol_xdt,
        img_xds,
        imaging_weights_params={
            "weighting": "briggs",
            "robust": robust,
            "casa_weighting_implementation": True,
        },
        ms_data_group_in_name="base",
        ms_data_group_out_name="av" + str(robust),
        ms_data_group_out_modified={"weight_imaging": "WEIGHT_AV_R_" + str(robust)},
        overwrite=True,
    )

ps_single_pol_xdt["twhya_selfcal_5chans_lsrk_xx_0"]

## Plotting code

In [ ]:
def plot_astroviper_vs_casa_weights_interactive(
    ps_xdt, ms_xdt_name, robust, freq_idx, bl_idx
):
    import matplotlib.pyplot as plt
    import numpy as np

    ms_xdt = ps_xdt[ms_xdt_name]
    av = ms_xdt[f"WEIGHT_AV_R_{robust}"].isel(
        baseline_id=bl_idx, frequency=freq_idx, polarization=0
    )
    casa = ms_xdt[f"WEIGHT_CASA_R_{robust}"].isel(
        baseline_id=bl_idx, frequency=freq_idx
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10), sharex=True)

    ax1.plot(av, label="AstroVIPER WEIGHT")
    ax1.plot(casa, label="CASA WEIGHT")

    if robust == 2:
        if ms_xdt[f"WEIGHT"].shape[-1] == 2:
            natural_weights = ms_xdt[f"WEIGHT"].isel(
                baseline_id=bl_idx, frequency=freq_idx
            )
            natural_weights = (natural_weights[..., 0] + natural_weights[..., 1]) / 2
        else:
            natural_weights = ms_xdt[f"WEIGHT"].isel(
                baseline_id=bl_idx, frequency=freq_idx, polarization=0
            )

        ax1.plot(natural_weights, label="Natural WEIGHT", linestyle="--")

    ax1.set(
        ylabel="Imaging Weight",
        title=f"R={robust} | freq={freq_idx} | baseline={bl_idx}",
    )

    per_dif = 100 * (av - casa) / (casa.max())
    ax2.plot(per_dif, label="Percentage Difference (AstroVIPER - CASA)")
    ax2.set(xlabel="Time Index", ylabel="Percentage Difference")

    print("The max difference is:", np.nanmax(per_dif))

    ax1.grid(True)
    ax1.legend()
    plt.show()
    plt.close(fig)


def plot_astroviper_vs_casa_weights_imshow_interactive(
    ps_xdt, ms_xdt_name, robust, bl_idx
):
    import matplotlib.pyplot as plt
    import numpy as np

    ms_xdt = ps_xdt[ms_xdt_name]
    av = ms_xdt[f"WEIGHT_AV_R_{robust}"].isel(baseline_id=bl_idx, polarization=0).values
    casa = ms_xdt[f"WEIGHT_CASA_R_{robust}"].isel(baseline_id=bl_idx).values
    av[av == 0] = np.nan
    casa[casa == 0] = np.nan

    per_dif = 100 * (av - casa) / (np.nanmax(casa))

    # Wide figure for horizontal layout
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 8), constrained_layout=True)

    im_kwargs = dict(aspect="auto", interpolation="nearest")

    # Plot AstroVIPER
    im1 = ax1.imshow(av, **im_kwargs)
    ax1.set_box_aspect(1)
    ax1.set_title(f"AstroVIPER | R={robust} | baseline={bl_idx}")
    ax1.set_ylabel("Imaging Weight")
    ax1.set_xlabel("Frequency Index")
    ax1.set_ylabel("Time Index")
    fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)

    # Plot CASA
    im2 = ax2.imshow(casa, **im_kwargs)
    ax2.set_box_aspect(1)
    ax2.set_title(f"CASA | R={robust} | baseline={bl_idx}")
    ax2.set_ylabel("Imaging Weight")
    ax2.set_xlabel("Frequency Index")
    ax2.set_ylabel("Time Index")
    fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)

    # Plot % difference
    im3 = ax3.imshow(per_dif, **im_kwargs)
    ax3.set_box_aspect(1)
    ax3.set_title("Percentage Difference (AstroVIPER - CASA)")
    ax3.set_xlabel("Frequency Index")
    ax3.set_ylabel("Time Index")
    fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04)

    print("The max difference is:", float(np.nanmax(per_dif)))

    plt.show()
    plt.close(fig)


def plot_astroviper_vs_casa_weights_max_interactive(ps_xdt, ms_xdt_name):
    import warnings

    import matplotlib.pyplot as plt
    import numpy as np

    with warnings.catch_warnings():
        warnings.filterwarnings(
            "ignore", category=RuntimeWarning, message="All-NaN axis encountered"
        )
        ms_xdt = ps_xdt[ms_xdt_name]
        n_baseline = ms_xdt.dims["baseline_id"]
        robust_values = [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]
        max_values = np.zeros((n_baseline, len(robust_values)))
        max_of_max = 0
        max_of_max_index = (0, 0, 0, 0)  # time, baseline, frequency, robust index
        for i, robust in enumerate(robust_values):
            av = ms_xdt[f"WEIGHT_AV_R_{robust}"].isel(polarization=0)
            casa = ms_xdt[f"WEIGHT_CASA_R_{robust}"]
            per_dif = 100 * (av - casa) / (casa.max())

            max_values[:, i] = np.nanmax(
                per_dif, axis=(0, 2)
            )  # Find max in time and frequency for each baseline

            max_of_max_index_temp = np.nanargmax(per_dif)
            max_of_max_index_temp = np.unravel_index(
                max_of_max_index_temp, per_dif.shape
            )
            max_of_max_temp = per_dif[max_of_max_index_temp]

            if max_of_max_temp > max_of_max:
                max_of_max = max_of_max_temp
                max_of_max_index = max_of_max_index_temp + (
                    i,
                )  # Add robust index to tuple

    print(
        f"The overall max percentage difference is {max_of_max.values:.5f}% for time {max_of_max_index[0]}, baseline {max_of_max_index[1]}, frequency {max_of_max_index[2]}, robust {robust_values[max_of_max_index[3]]}"
    )
    fig = plt.figure(figsize=(10, 6))
    for i, robust in enumerate(robust_values):
        plt.plot(max_values[:, i], label=f"R={robust}")
    plt.xlabel("Baseline Index")
    plt.ylabel("Max Percentage Difference (AstroVIPER - CASA)")
    plt.title("Max Percentage Difference per Baseline for Different Robust Values")
    plt.legend()
    plt.grid(True)
    plt.show()
    plt.close(fig)

### Compare CASA and AstroVIPER

CASA flags the first channel.

In [ ]:
# If not installed:
# !pip install ipywidgets
# %matplotlib widget
import ipywidgets as widgets
from ipywidgets import fixed, interact

robust_options = [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]

interact(
    plot_astroviper_vs_casa_weights_interactive,
    ps_xdt=fixed(ps_single_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_single_pol_xdt.keys())),
    robust=widgets.Dropdown(options=robust_options, value=-2, description="robust"),
    freq_idx=widgets.IntSlider(min=0, max=4, step=1, value=2, description="freq_chan"),
    bl_idx=widgets.IntSlider(
        min=0, max=170, step=1, value=10, description="baseline_id"
    ),
)

In [ ]:
# %matplotlib widget
import ipywidgets as widgets
from ipywidgets import fixed, interact

robust_options = [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]

interact(
    plot_astroviper_vs_casa_weights_imshow_interactive,
    ps_xdt=fixed(ps_single_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_single_pol_xdt.keys())),
    robust=widgets.Dropdown(options=robust_options, value=-2, description="robust"),
    bl_idx=widgets.IntSlider(
        min=0, max=170, step=1, value=10, description="baseline_id"
    ),
)

In [ ]:
import ipywidgets as widgets
from ipywidgets import fixed, interact

interact(
    plot_astroviper_vs_casa_weights_max_interactive,
    ps_xdt=fixed(ps_single_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_single_pol_xdt.keys())),
)

## Dual Pol XX, YY

In [ ]:
from toolviper.utils.data import download, update
from xradio.measurement_set import open_processing_set

update()
download(file="twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr")
ps_dual_pol_xdt = open_processing_set(
    "twhya_selfcal_5chans_lsrk_compare_weights.ps.zarr"
)
ps_dual_pol_xdt.xr_ps.summary()

In [ ]:
ps_dual_pol_xdt["twhya_selfcal_5chans_lsrk_0"]

### Create Weights With AstroVIPER

In [ ]:
import numpy as np
from xradio.image import make_empty_sky_image

from astroviper.processing_functions.imaging.calculate_imaging_weights import (
    calculate_imaging_weights,
)

image_size = [250, 250]
cell_size = np.array([-0.1, 0.1]) * np.pi / (180 * 3600)

# The empty sky image only defines the uv-grid geometry used to build the
# Briggs weight-density grid.  calculate_imaging_weights writes the imaging
# weights back into the processing set in place, one data group per robust
# value (out group "av<robust>", variable WEIGHT_AV_R_<robust>).
combined = ps_dual_pol_xdt.xr_ps.get_combined_field_and_source_xds()
phase_center = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=combined.attrs["center_field_name"]
).values
frequency_coords = ps_dual_pol_xdt.xr_ps.get_freq_axis().values
pol_coords = list(ps_dual_pol_xdt["twhya_selfcal_5chans_lsrk_0"].polarization.values)

img_xds = make_empty_sky_image(
    phase_center=phase_center,
    image_size=image_size,
    cell_size=cell_size,
    frequency_coords=frequency_coords,
    pol_coords=pol_coords,
    time_coords=[0],
    do_sky_coords=False,
)
img_xds.attrs["type"] = "image_dataset"

for robust in [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]:
    print("Calculating robust =", robust)
    calculate_imaging_weights(
        ps_dual_pol_xdt,
        img_xds,
        imaging_weights_params={
            "weighting": "briggs",
            "robust": robust,
            "casa_weighting_implementation": True,
        },
        ms_data_group_in_name="base",
        ms_data_group_out_name="av" + str(robust),
        ms_data_group_out_modified={"weight_imaging": "WEIGHT_AV_R_" + str(robust)},
        overwrite=True,
    )

ps_dual_pol_xdt

In [ ]:
# If not installed:
# !pip install ipywidgets
# %matplotlib widget
import ipywidgets as widgets
from ipywidgets import fixed, interact

robust_options = [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]

ms_dual_pol_xdt = ps_dual_pol_xdt["twhya_selfcal_5chans_lsrk_0"]

interact(
    plot_astroviper_vs_casa_weights_interactive,
    ps_xdt=fixed(ps_dual_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_dual_pol_xdt.keys())),
    robust=widgets.Dropdown(options=robust_options, value=-2, description="robust"),
    freq_idx=widgets.IntSlider(min=0, max=4, step=1, value=2, description="freq_chan"),
    bl_idx=widgets.IntSlider(
        min=0, max=170, step=1, value=10, description="baseline_id"
    ),
)

In [ ]:
# %matplotlib widget
import ipywidgets as widgets
from ipywidgets import fixed, interact

robust_options = [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]

ms_dual_pol_xdt = ps_dual_pol_xdt["twhya_selfcal_5chans_lsrk_0"]

interact(
    plot_astroviper_vs_casa_weights_imshow_interactive,
    ps_xdt=fixed(ps_dual_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_dual_pol_xdt.keys())),
    robust=widgets.Dropdown(options=robust_options, value=-2, description="robust"),
    bl_idx=widgets.IntSlider(
        min=0, max=170, step=1, value=10, description="baseline_id"
    ),
)

In [ ]:
import ipywidgets as widgets
from ipywidgets import fixed, interact

interact(
    plot_astroviper_vs_casa_weights_max_interactive,
    ps_xdt=fixed(ps_dual_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_dual_pol_xdt.keys())),
)

## Full Pol XX, XY, YX, YY

In [ ]:
from toolviper.utils.data import download, update
from xradio.measurement_set import open_processing_set

update()
download(file="3c286_Band6_5chans_lsrk_compare_weights.ps.zarr")

ps_full_pol_xdt = open_processing_set("3c286_Band6_5chans_lsrk_compare_weights.ps.zarr")
ps_full_pol_xdt.xr_ps.summary()

In [ ]:
ps_full_pol_xdt["3c286_Band6_5chans_lsrk_0"]

### Create Weights With AstroVIPER

In [ ]:
import numpy as np
from xradio.image import make_empty_sky_image

from astroviper.processing_functions.imaging.calculate_imaging_weights import (
    calculate_imaging_weights,
)

image_size = [250, 250]
cell_size = np.array([-0.1, 0.1]) * np.pi / (180 * 3600)

# The empty sky image only defines the uv-grid geometry used to build the
# Briggs weight-density grid.  calculate_imaging_weights writes the imaging
# weights back into the processing set in place, one data group per robust
# value (out group "av<robust>", variable WEIGHT_AV_R_<robust>).
combined = ps_full_pol_xdt.xr_ps.get_combined_field_and_source_xds()
phase_center = combined.FIELD_PHASE_CENTER_DIRECTION.sel(
    field_name=combined.attrs["center_field_name"]
).values
frequency_coords = ps_full_pol_xdt.xr_ps.get_freq_axis().values
pol_coords = list(ps_full_pol_xdt["3c286_Band6_5chans_lsrk_0"].polarization.values)

img_xds = make_empty_sky_image(
    phase_center=phase_center,
    image_size=image_size,
    cell_size=cell_size,
    frequency_coords=frequency_coords,
    pol_coords=pol_coords,
    time_coords=[0],
    do_sky_coords=False,
)
img_xds.attrs["type"] = "image_dataset"

for robust in [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]:
    print("Calculating robust =", robust)
    calculate_imaging_weights(
        ps_full_pol_xdt,
        img_xds,
        imaging_weights_params={
            "weighting": "briggs",
            "robust": robust,
            "casa_weighting_implementation": True,
        },
        ms_data_group_in_name="base",
        ms_data_group_out_name="av" + str(robust),
        ms_data_group_out_modified={"weight_imaging": "WEIGHT_AV_R_" + str(robust)},
        overwrite=True,
    )

ms_full_pol_xdt = ps_full_pol_xdt["3c286_Band6_5chans_lsrk_0"]

In [ ]:
ms_full_pol_xdt["WEIGHT_AV_R_-2"]

In [ ]:
# If not installed:
# !pip install ipywidgets
# %matplotlib widget
import ipywidgets as widgets
from ipywidgets import fixed, interact

robust_options = [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]


interact(
    plot_astroviper_vs_casa_weights_interactive,
    ps_xdt=fixed(ps_full_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_full_pol_xdt.keys())),
    robust=widgets.Dropdown(options=robust_options, value=-2, description="robust"),
    freq_idx=widgets.IntSlider(min=0, max=4, step=1, value=2, description="freq_chan"),
    bl_idx=widgets.IntSlider(
        min=0, max=170, step=1, value=10, description="baseline_id"
    ),
)

In [ ]:
# %matplotlib widget
import ipywidgets as widgets
from ipywidgets import fixed, interact

robust_options = [-2, -1.45, -0.5, 0, 0.5, 1.33, 2]

interact(
    plot_astroviper_vs_casa_weights_imshow_interactive,
    ps_xdt=fixed(ps_full_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_full_pol_xdt.keys())),
    robust=widgets.Dropdown(options=robust_options, value=-2, description="robust"),
    bl_idx=widgets.IntSlider(
        min=0, max=170, step=1, value=10, description="baseline_id"
    ),
)

In [ ]:
import ipywidgets as widgets
from ipywidgets import fixed, interact

interact(
    plot_astroviper_vs_casa_weights_max_interactive,
    ps_xdt=fixed(ps_full_pol_xdt),
    ms_xdt_name=widgets.Dropdown(options=list(ps_full_pol_xdt.keys())),
)